# Understand the Instacart data

The project asks: **which products should Instacart recommend to help customers build their next grocery basket?**

Before looking for findings, I'll get comfortable with the tables. Today I only need to explain an order and a product line.

In [2]:
from pathlib import Path
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'03_Code/01_Data_Preparation/prepare_data.py').exists()),None)
assert ROOT is not None, 'Open the notebook from the Instacart Basket Analysis folder.'
DATA=ROOT/'02_Data/02_Prepared'
print('Project folder:',ROOT)

ModuleNotFoundError: No module named 'pandas'

## 1. Look at the products in one order

The small learning file follows one selected user. It helps explain the structure; it isn't a representative sample of Instacart shoppers.

In [ ]:
items=pd.read_csv(DATA/'learning_order_items.csv')
first_order=items.loc[items.order_number.eq(1)]
first_order[['user_id','order_id','product_name','add_to_cart_order','reordered']]

**First question:** These rows share one order ID. Are they separate orders, or products inside the same order?

Answer: [I'll fill this in after the discussion.]

I can stop here for the first lesson.

## 2. Follow the order sequence

Now I'll look at the orders from the same user. The order number gives the sequence. It isn't a date or a basket size.

In [ ]:
orders=pd.read_csv(DATA/'learning_orders.csv')
orders[['user_id','order_id','order_number','eval_set','product_lines','repeat_product_lines']]

## 3. What does reordered mean?

A value of 1 means the user has bought that product before. It doesn't have to be in the immediately previous order. I'll compare the first two baskets to see an example.

In [ ]:
items.loc[items.order_number.le(2),['order_number','product_name','reordered']]

**Question:** If a basket has ten product lines and six have reordered = 1, what's the repeat-line share?

Answer: [To discuss.]

That share doesn't mean six out of ten customers returned. It's measuring product lines.

## 4. Why are some later baskets unknown?

The original release includes final orders labelled train or test. Train contents are provided; test contents aren't. A blank test basket isn't an empty basket.

In [ ]:
import duckdb
with duckdb.connect(str(DATA/'instacart.duckdb'),read_only=True) as con:
    example=con.execute("SELECT order_id,eval_set,contents_available,product_lines FROM order_summary WHERE eval_set='test' ORDER BY order_id LIMIT 3").df()
example

## 5. Keep the answer out of the input

For the later recommendation task, earlier orders are the history. A held-back later basket is the answer to check against. If I use that answer to build the list, the comparison won't be fair.

The next cell shows the prepared cutoffs for one labelled user. No predictions have been made.

In [ ]:
plan=pd.read_csv(DATA/'evaluation_plan.csv')
plan.loc[plan.user_id.eq(1)]

## Next lesson

I'll go through the data explanation and metric rules with the tutor. Then we'll explore repeat purchasing and decide what comparisons are fair. The findings stay open until we've discussed the evidence.

# Data quality practice

## 1. Inspect missing categories

This check reads the prepared database without changing it. We're counting **products**, not orders, purchases or customers.

- `SELECT` chooses what to show.
- `COUNT(*)` counts the product rows.
- `WHERE` keeps only the literal department label `missing`.
- `GROUP BY` gives one count for that label.

Select the code cell below and press **Shift + Enter**. Use this project's `.venv` Python environment if VS Code asks for a kernel.

In [1]:
from pathlib import Path
import duckdb

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / '03_Code' / '01_Data_Preparation' / 'prepare_data.py').exists())
DB = ROOT / '02_Data' / '02_Prepared' / 'instacart.duckdb'

query = """
SELECT department, COUNT(*) AS products
FROM product_catalog
WHERE department = 'missing'
GROUP BY department
"""

with duckdb.connect(str(DB), read_only=True) as con:
    result = con.execute(query).df()

result

,department,products
0,missing,1258


## Read the result

What does the count describe? Should products with an unknown department be deleted, kept with the missing label, or given an explicit Unknown group in a reporting view?

We haven't changed any data. We'll make that decision together after reading the output.